# Stardog Quick Start Tutorial

This notebook walks you through working with **Stardog Cloud from Python**, step by step:

1. Connect to a Stardog server and check it is alive
2. Understand databases, named graphs and triples
3. Load RDF data inside a transaction
4. Query it with SPARQL: `SELECT`, `ASK`, `CONSTRUCT`, aggregations, property paths
5. Change data with SPARQL Update
6. Undo changes with a transaction rollback
7. Look at a query plan and export data
8. Clean up safely

Everything runs against the database named in your `.env` file (`STARDOG_DATABASE`). The notebook writes **only** to its own named graph `urn:QuickStart:data`, so other data in the database (recipe model, supply chain, Designer models) is never touched.

**Before you start:** select the project's `.venv` as the notebook kernel (VS Code: *Select Kernel → Python Environments → .venv*).

## 1. Key concepts in 60 seconds

**RDF triples.** Stardog stores data as *triples*: `subject → predicate → object`, for example

```
ex:alice   ex:worksFor   ex:acme .
ex:alice   ex:name       "Alice" .
```

Subjects and predicates are IRIs (global identifiers). Objects are either IRIs (links to other things) or literals (text, numbers, dates). A set of triples forms a **graph**.

**Database.** A Stardog server holds several databases. Stardog Cloud plans limit how many you can create, so this tutorial reuses an existing one.

**Named graphs.** Inside a database, triples can be grouped into *named graphs*, each identified by an IRI. Named graphs let several projects share one database without mixing their data. Triples that are not in any named graph live in the *default graph*.

**SPARQL** is the query language for RDF, like SQL is for tables.

## 2. Setup: credentials and connection details

Credentials live in the `.env` file at the project root, never in code:

```
STARDOG_ENDPOINT=https://<your-endpoint>.stardog.cloud:5820
STARDOG_USERNAME=<server user>
STARDOG_PASSWORD=<password>
STARDOG_DATABASE=mysampledb
```

> **Common pitfall:** `STARDOG_USERNAME` must be a user created **on the Stardog server** (Studio → Security → Users). Your cloud.stardog.com login email does not work here and gives `401 Unauthorized`.

`load_dotenv()` searches upwards from the current folder, so it finds the `.env` in the project root.

In [1]:
import os

import pandas as pd
import stardog
from dotenv import load_dotenv

load_dotenv()

conn_details = {
    'endpoint': os.environ['STARDOG_ENDPOINT'],
    'username': os.environ['STARDOG_USERNAME'],
    'password': os.environ['STARDOG_PASSWORD'],
}
DB_NAME = os.environ['STARDOG_DATABASE']
GRAPH = 'urn:QuickStart:data'   # this tutorial's own named graph

print('Endpoint:', conn_details['endpoint'])
print('User:    ', conn_details['username'])
print('Database:', DB_NAME)

Endpoint: https://sd-fd55b7e2.stardog.cloud:5820
User:     rakesh
Database: mysampledb


## 3. Talk to the server with `stardog.Admin`

`stardog.Admin` is for **server-level** tasks: checking the server, listing and creating databases, managing users and settings. Creating it also checks the server is reachable and your login works, so this is the first thing that fails if credentials are wrong.

In [2]:
with stardog.Admin(**conn_details) as admin:
    print('Server alive:', admin.alive())
    print('Databases:   ', [db.name for db in admin.databases()])

Server alive: True


Databases:    ['catalog', 'querylog', 'mysampledb']


`catalog` and `querylog` are system databases that Stardog Cloud creates. Your own databases appear next to them.

### Creating a database (optional)

On a paid plan or local server you would create a database like this. It is switched off here because the Stardog Cloud free plan allows only one user database and it is already taken by `STARDOG_DATABASE`.

In [3]:
CREATE_DEMO_DB = False   # set to True only if your plan allows another database

if CREATE_DEMO_DB:
    with stardog.Admin(**conn_details) as admin:
        db = admin.new_database('quickstart_demo')
        print('Created', db.name)
        db.drop()          # delete it again
        print('Dropped', db.name)
else:
    print('Skipped database creation')

Skipped database creation


## 4. Open a connection to the database

`stardog.Connection` is for **database-level** work: loading data, queries, updates. We keep one connection open for the whole notebook and close it at the end.

We also define a small helper, `select_df`, that runs a `SELECT` query and returns a pandas DataFrame. Stardog returns results in the standard SPARQL JSON format:

```
{'head': {'vars': ['name', 'age']},
 'results': {'bindings': [{'name': {'type': 'literal', 'value': 'Alice'}, ...}]}}
```

The helper just pulls out the `value` of each variable.

In [4]:
conn = stardog.Connection(DB_NAME, **conn_details)

PREFIXES = '''
PREFIX ex:  <http://example.org/quickstart#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
'''

def select_df(query, **kwargs):
    result = conn.select(PREFIXES + query, **kwargs)
    columns = result['head']['vars']
    rows = [{c: b[c]['value'] if c in b else None for c in columns}
            for b in result['results']['bindings']]
    return pd.DataFrame(rows, columns=columns)

print('Connected to', DB_NAME, '- approx. total triples in database:', conn.size())

Connected to mysampledb - approx. total triples in database: 2510


## 5. Load data

### The data

A small company network written in **Turtle**, the most readable RDF format:

- `a` is shorthand for `rdf:type` (the *class* of a thing)
- `;` repeats the subject for the next predicate
- `,` repeats subject and predicate for the next object
- `"30"^^xsd:integer` is a typed literal (a number, not text)

In [5]:
DATA = b'''
@prefix ex:  <http://example.org/quickstart#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:acme    a ex:Company ; ex:name "Acme Corp"  ; ex:city "Bengaluru" .
ex:globex  a ex:Company ; ex:name "Globex Ltd" ; ex:city "Pune" .
ex:initech a ex:Company ; ex:name "Initech"    ; ex:city "Hyderabad" .

ex:alice a ex:Person ; ex:name "Alice" ; ex:age "30"^^xsd:integer ; ex:worksFor ex:acme ;
         ex:role "Data Engineer"  ; ex:knows ex:bob, ex:carol .
ex:bob   a ex:Person ; ex:name "Bob"   ; ex:age "35"^^xsd:integer ; ex:worksFor ex:acme ;
         ex:role "Manager"        ; ex:knows ex:dave .
ex:carol a ex:Person ; ex:name "Carol" ; ex:age "28"^^xsd:integer ; ex:worksFor ex:globex ;
         ex:role "Data Scientist" ; ex:knows ex:alice .
ex:dave  a ex:Person ; ex:name "Dave"  ; ex:age "42"^^xsd:integer ; ex:worksFor ex:initech ;
         ex:role "Architect" .
ex:eve   a ex:Person ; ex:name "Eve"   ; ex:age "25"^^xsd:integer ; ex:worksFor ex:globex ;
         ex:role "Analyst"        ; ex:knows ex:carol, ex:dave .
'''

### Load it in a transaction

Writes in Stardog happen inside a **transaction**: `begin()` → changes → `commit()`. Either all changes are saved or none are.

- `stardog.content.Raw` wraps data held in memory. For files use `stardog.content.File('path.ttl')`.
- `graph_uri=GRAPH` puts the triples into our named graph instead of the default graph.
- We first `clear(graph_uri=GRAPH)` so re-running the notebook doesn't mix old and new data.

> ⚠️ **Always pass `graph_uri` to `clear()`.** Called without it, `clear()` deletes **every triple in the database**.

In [6]:
conn.begin()
conn.clear(graph_uri=GRAPH)
conn.add(stardog.content.Raw(DATA, content_type='text/turtle'), graph_uri=GRAPH)
conn.commit()

count = select_df(f'SELECT (COUNT(*) AS ?triples) FROM <{GRAPH}> {{ ?s ?p ?o }}')
print('Triples in', GRAPH, ':', count['triples'][0])

Triples in urn:QuickStart:data : 40


## 6. Query with SPARQL `SELECT`

A `SELECT` query matches **triple patterns**: triples with variables (`?person`) in them. Every way the variables can be filled in from the data becomes one result row.

`FROM <urn:QuickStart:data>` tells Stardog to query our named graph. Without it, Stardog only looks at the default graph.

In [7]:
select_df(f'''
SELECT ?name ?role ?age
FROM <{GRAPH}>
WHERE {{
  ?person a ex:Person ;
          ex:name ?name ;
          ex:role ?role ;
          ex:age  ?age .
}}
ORDER BY ?name
''')

,name,role,age
0,Alice,Data Engineer,30
1,Bob,Manager,35
2,Carol,Data Scientist,28
3,Dave,Architect,42
4,Eve,Analyst,25


### Joins: following links between things

Patterns that share a variable are joined automatically. Here `?company` links each person to their employer, so we can read the company's name and city. No `JOIN` keyword is needed.

In [8]:
select_df(f'''
SELECT ?person ?company ?city
FROM <{GRAPH}>
WHERE {{
  ?p ex:name ?person ; ex:worksFor ?c .
  ?c ex:name ?company ; ex:city ?city .
}}
ORDER BY ?company ?person
''')

,person,company,city
0,Alice,Acme Corp,Bengaluru
1,Bob,Acme Corp,Bengaluru
2,Carol,Globex Ltd,Pune
3,Eve,Globex Ltd,Pune
4,Dave,Initech,Hyderabad


### Filters

`FILTER` keeps only rows that satisfy a condition. Because the ages were loaded as `xsd:integer`, comparisons are numeric.

In [9]:
select_df(f'''
SELECT ?name ?age
FROM <{GRAPH}>
WHERE {{
  ?p a ex:Person ; ex:name ?name ; ex:age ?age .
  FILTER(?age >= 30)
}}
ORDER BY DESC(?age)
''')

,name,age
0,Dave,42
1,Bob,35
2,Alice,30


### Aggregation

`GROUP BY` with `COUNT`, `AVG`, `GROUP_CONCAT` works much like SQL.

In [10]:
select_df(f'''
SELECT ?company
       (COUNT(?p) AS ?employees)
       (AVG(?age) AS ?avg_age)
       (GROUP_CONCAT(?name; separator=", ") AS ?people)
FROM <{GRAPH}>
WHERE {{
  ?p ex:worksFor ?c ; ex:name ?name ; ex:age ?age .
  ?c ex:name ?company .
}}
GROUP BY ?company
ORDER BY DESC(?employees)
''')

,company,employees,avg_age,people
0,Globex Ltd,2,26.5,"Eve, Carol"
1,Acme Corp,2,32.5,"Bob, Alice"
2,Initech,1,42,Dave


### OPTIONAL: keep rows even when data is missing

Dave knows nobody. A plain pattern would drop him; `OPTIONAL` keeps him with an empty value.

In [11]:
select_df(f'''
SELECT ?name (COUNT(?friend) AS ?knows_count)
FROM <{GRAPH}>
WHERE {{
  ?p a ex:Person ; ex:name ?name .
  OPTIONAL {{ ?p ex:knows ?friend }}
}}
GROUP BY ?name
ORDER BY DESC(?knows_count)
''')

,name,knows_count
0,Eve,2
1,Alice,2
2,Carol,1
3,Bob,1
4,Dave,0


### Property paths: multi-hop traversal

This is where graphs beat tables. `ex:knows+` means "follows `knows` **one or more** times", so we get everyone Eve can reach through her network, however many hops away. In SQL this needs a recursive CTE.

In [12]:
select_df(f'''
SELECT DISTINCT ?reachable
FROM <{GRAPH}>
WHERE {{
  ex:eve ex:knows+ ?r .
  ?r ex:name ?reachable .
  FILTER(?r != ex:eve)
}}
ORDER BY ?reachable
''')

,reachable
0,Alice
1,Bob
2,Carol
3,Dave


## 7. `ASK` and `CONSTRUCT` queries

- **`ASK`** returns just `True` or `False`: does any match exist?
- **`CONSTRUCT`** returns *new triples* built from a template. It's handy for reshaping data or deriving new links.

In [13]:
has_architect = conn.ask(PREFIXES + f'ASK FROM <{GRAPH}> {{ ?p ex:role "Architect" }}')
has_ceo = conn.ask(PREFIXES + f'ASK FROM <{GRAPH}> {{ ?p ex:role "CEO" }}')
print('Is there an Architect?', has_architect)
print('Is there a CEO?       ', has_ceo)

Is there an Architect? True
Is there a CEO?        False


The `CONSTRUCT` below derives a new relationship, `ex:colleagueOf`, between people at the same company. The result is RDF (Turtle bytes); it is **not** saved to the database unless you load it.

In [14]:
turtle = conn.graph(PREFIXES + f'''
CONSTRUCT {{ ?a ex:colleagueOf ?b }}
FROM <{GRAPH}>
WHERE {{
  ?a ex:worksFor ?c . ?b ex:worksFor ?c .
  FILTER(?a != ?b)
}}
''')
print(turtle.decode())

@prefix ex: <http://example.org/quickstart#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:colleagueOf ex:bob .
ex:bob ex:colleagueOf ex:alice .
ex:carol ex:colleagueOf ex:eve .
ex:eve ex:colleagueOf ex:carol .


## 8. Change data with SPARQL Update

SPARQL Update modifies data with a query:

- `INSERT DATA` adds fixed triples
- `DELETE ... INSERT ... WHERE` finds matches and rewrites them (like SQL `UPDATE`)

Each `conn.update()` call outside an explicit transaction is committed immediately. Note the `GRAPH <...>` blocks, which target our named graph.

In [15]:
# Add a new employee
conn.update(PREFIXES + f'''
INSERT DATA {{
  GRAPH <{GRAPH}> {{
    ex:frank a ex:Person ; ex:name "Frank" ; ex:age 31 ;
             ex:role "Data Engineer" ; ex:worksFor ex:initech ; ex:knows ex:dave .
  }}
}}
''')

# Promote Bob: replace his role
conn.update(PREFIXES + f'''
DELETE {{ GRAPH <{GRAPH}> {{ ex:bob ex:role ?old }} }}
INSERT {{ GRAPH <{GRAPH}> {{ ex:bob ex:role "Director" }} }}
WHERE  {{ GRAPH <{GRAPH}> {{ ex:bob ex:role ?old }} }}
''')

select_df(f'''
SELECT ?name ?role ?company FROM <{GRAPH}>
WHERE {{ ?p ex:name ?name ; ex:role ?role ; ex:worksFor/ex:name ?company }}
ORDER BY ?name
''')

,name,role,company
0,Alice,Data Engineer,Acme Corp
1,Bob,Director,Acme Corp
2,Carol,Data Scientist,Globex Ltd
3,Dave,Architect,Initech
4,Eve,Analyst,Globex Ltd
5,Frank,Data Engineer,Initech


`ex:worksFor/ex:name` in that last query is another property path: "follow `worksFor`, then `name`". It skips the intermediate company variable.

## 9. Transactions: rollback

If something goes wrong halfway, `rollback()` throws away every change made since `begin()`. Below we delete all people, check they are gone *inside* the transaction, then roll back.

In [16]:
def count_people():
    return int(select_df(f'SELECT (COUNT(?p) AS ?n) FROM <{GRAPH}> {{ ?p a ex:Person }}')['n'][0])

print('People before:', count_people())

conn.begin()
conn.update(PREFIXES + f'DELETE WHERE {{ GRAPH <{GRAPH}> {{ ?p a ex:Person }} }}')
print('People inside the transaction (after delete):', count_people())
conn.rollback()

print('People after rollback:', count_people())

People before: 6


People inside the transaction (after delete): 0


People after rollback: 6


Queries run on the same connection see the transaction's uncommitted changes. Other users would still see the old data until `commit()`.

## 10. Query plans with `explain()`

`explain()` shows how Stardog will execute a query: the order of joins, index scans and estimated cardinalities. It's useful for tuning slow queries.

In [17]:
print(conn.explain(PREFIXES + f'''
SELECT ?person ?company FROM <{GRAPH}>
WHERE {{ ?p ex:name ?person ; ex:worksFor ?c . ?c ex:name ?company }}
'''))

prefix ex: <http://example.org/quickstart#>
prefix xsd: <http://www.w3.org/2001/XMLSchema#>

From <urn:QuickStart:data>
Projection(?person, ?company) [#1]
`─ MergeJoin(?c) [#1]
   +─ Scan[PSOC](?c, ex:name, ?company) [#1]
   `─ Sort(?c) [#1]
      `─ Restriction(?person, ?c) [#1]
         `─ MergeJoin(?p) [#1]
            +─ Scan[PSOC](?p, ex:name, ?person) [#1]
            `─ Scan[PSOC](?p, ex:worksFor, ?c) [#1]



## 11. Export data

`export()` dumps a graph in any RDF format. Here we save our named graph as Turtle next to this notebook.

In [18]:
exported = conn.export(content_type='text/turtle', graph_uri=GRAPH)
with open('quick_start_export.ttl', 'wb') as f:
    f.write(exported)

print(f'Wrote quick_start_export.ttl ({len(exported)} bytes). First lines:\n')
print('\n'.join(exported.decode().splitlines()[:12]))

Wrote quick_start_export.ttl (3300 bytes). First lines:

@prefix Recipe_Basic: <urn:Recipe_Basic:> .
@prefix : <http://api.stardog.com/> .
@prefix stardog: <tag:stardog:api:> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix so: <https://schema.org/> .

<http://example.org/quickstart#acme> a <http://example.org/quickstart#Company> ;
   <http://example.org/quickstart#name> "Acme Corp" ;
   <http://example.org/quickstart#city> "Bengaluru" .


## 12. Clean up

Remove this tutorial's named graph (and only that graph) and close the connection. Set `CLEAN_UP = False` to keep the data so you can explore it in Stardog Studio. In Studio, add `FROM <urn:QuickStart:data>` to your queries.

In [19]:
CLEAN_UP = True

if CLEAN_UP:
    conn.begin()
    conn.clear(graph_uri=GRAPH)   # graph_uri is essential, see section 5
    conn.commit()
    print('Removed', GRAPH)

conn.close()
print('Connection closed')

Removed urn:QuickStart:data
Connection closed


## Summary

| Task | Python |
|---|---|
| Server tasks (alive, list/create DBs) | `stardog.Admin(**conn_details)` |
| Database work | `stardog.Connection(db, **conn_details)` |
| Load data | `begin()` → `add(Raw(...) or File(...), graph_uri=...)` → `commit()` |
| Read | `select()`, `ask()`, `graph()` (CONSTRUCT), `paths()` |
| Write with SPARQL | `update()` |
| Undo | `rollback()` |
| Performance | `explain()` |
| Dump | `export(graph_uri=...)` |
| Delete a graph | `clear(graph_uri=...)` (**never without `graph_uri`**) |

**Next steps**

- `recipes/create_model.py`: build an OWL model and let Stardog **reason** new facts (recipe → allergen).
- `supply_chain/supply_chain_tutorial.ipynb`: run **graph algorithms** (PageRank, communities, …) on a logistics network.